## Linear Probing in Machine Unlearning

This paper tires to reproduce the emprical finding of the paper "An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations" [https://arxiv.org/pdf/2604.08271] by Yichen Gao et al presented at AISTATS 2026. 

### Findings by the paper:
1. The paper shows that most standard unlearning methods (including Gradient Ascent / NegGrad+, Random Label, SalUn, SCRUB, etc.) only create a superficial "illusion" of unlearning.
2. They mainly cause feature–classifier misalignment:
    - The backbone features for the forget class remain highly discriminative and linearly separable.
    - The unlearning mostly messes up the final classification layer's weights for that class.
    - Therefore, when you freeze the backbone and train a new linear classifier (linear probing) on the full training set (including forget samples), you recover near-original forget accuracy. [which is obvious]
3. They also show that even the "gold standard" (retrain from scratch on retain only) still has somewhat separable forget features due to representation transferability. [which is obvious]

In [3]:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.models import resnet18
from scipy.sparse import load_npz

from tqdm import tqdm
import os
import json
from models import get_model
import warnings
warnings.filterwarnings("ignore")

In [4]:
def evaluate_accuracy(model,test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    acc = 100. * correct / total
    return round(acc,3)

In [6]:
batch_size = 128
NUM_WORKERS = 4

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5071, 0.4867, 0.4408],
        std=[0.2675, 0.2565, 0.2761]
    )
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5071, 0.4867, 0.4408],
        std=[0.2675, 0.2565, 0.2761]
    )
])

train_dataset = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=transform_train
)

test_dataset = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=transform_test
)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS
)

In [7]:
import random
import numpy as np
from torch.utils.data import DataLoader, Subset

# -----------------------------
# Randomly choose a class to forget
# -----------------------------
num_classes = 100
forget_class = random.randint(0, num_classes - 1)

print(f"Chosen forget class: {forget_class}")

# -----------------------------
# Get targets
# CIFAR100 stores labels in .targets
# -----------------------------
train_targets = np.array(train_dataset.targets)
test_targets = np.array(test_dataset.targets)

# -----------------------------
# Split indices
# retain = everything except forget_class
# forget = only forget_class
# -----------------------------
train_forget_idx = np.where(train_targets == forget_class)[0]
train_retain_idx = np.where(train_targets != forget_class)[0]

test_forget_idx = np.where(test_targets == forget_class)[0]
test_retain_idx = np.where(test_targets != forget_class)[0]

# -----------------------------
# Create subsets
# -----------------------------
train_forget_dataset = Subset(train_dataset, train_forget_idx)
train_retain_dataset = Subset(train_dataset, train_retain_idx)

test_forget_dataset = Subset(test_dataset, test_forget_idx)
test_retain_dataset = Subset(test_dataset, test_retain_idx)

# -----------------------------
# Create loaders
# -----------------------------
train_forget_loader = DataLoader(
    train_forget_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS
)

train_retain_loader = DataLoader(
    train_retain_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS
)

test_forget_loader = DataLoader(
    test_forget_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_retain_loader = DataLoader(
    test_retain_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS
)

# -----------------------------
# Print stats
# -----------------------------
print(f"Train retain samples: {len(train_retain_dataset)}")
print(f"Train forget samples: {len(train_forget_dataset)}")

print(f"Test retain samples: {len(test_retain_dataset)}")
print(f"Test forget samples: {len(test_forget_dataset)}")

Chosen forget class: 7
Train retain samples: 49500
Train forget samples: 500
Test retain samples: 9900
Test forget samples: 100


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_model(model_name='resnet18',num_classes=100)
model = model.to(device)

checkpoint_path = "./CIFAR100_ResNet18_100Epochs.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device)['model_state_dict'])

<All keys matched successfully>

In [10]:
print("-------------------Before Unlearning-------------")
print("Forget Train Accuracy::",evaluate_accuracy(model,train_forget_loader))
print("Retain Train Accuracy::",evaluate_accuracy(model,train_retain_loader))
print("Forget Test Accuracy::",evaluate_accuracy(model,test_forget_loader))
print("Retain Test Accuracy::",evaluate_accuracy(model,test_retain_loader))

-------------------Before Unlearning-------------


Forget Train Accuracy:: 100.0


Retain Train Accuracy:: 99.923


Forget Test Accuracy:: 65.0


Retain Test Accuracy:: 63.455


In [12]:
def GradientAscent(model,forget_loader,epochs=5,lr=1e-5,weight_decay=0):
    model.train()
    
    # -----------------------
    # Optimizer & loss
    # -----------------------
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )


    for epoch in tqdm(range(epochs)):
        running_loss = 0.0
        for inputs, targets in forget_loader:
            inputs, targets = inputs.to(device), targets.to(device)
    
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            (-loss).backward()
            optimizer.step()
    
            running_loss += loss.item()
        
        print(
            f"[GA] Epoch {epoch+1}/{epochs} | "
            f"Loss: {running_loss:.4f} | ")        
    return model

In [15]:
model.load_state_dict(torch.load(checkpoint_path, map_location=device)['model_state_dict'])
model = GradientAscent(model,train_forget_loader,epochs=3,lr=1e-5)

print("-------------------After Unlearning-------------")
print("Forget Train Accuracy::",evaluate_accuracy(model,train_forget_loader))
print("Retain Train Accuracy::",evaluate_accuracy(model,train_retain_loader))
print("Forget Test Accuracy::",evaluate_accuracy(model,test_forget_loader))
print("Retain Test Accuracy::",evaluate_accuracy(model,test_retain_loader))

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:00<00:00,  4.86it/s]

[GA] Epoch 1/3 | Loss: 35.9623 | 


 67%|██████▋   | 2/3 [00:00<00:00,  4.95it/s]

[GA] Epoch 2/3 | Loss: 38.7535 | 


...

[GA] Epoch 3/3 | Loss: 41.3916 | 
-------------------After Unlearning-------------


Forget Train Accuracy:: 2.4


Retain Train Accuracy:: 85.81


Forget Test Accuracy:: 2.0


Retain Test Accuracy:: 53.596


In [16]:
torch.save(model.state_dict(),'unlearned_resnet18_cifar100_0forgetclass_model.pth')

In [17]:
#Linear Probing

import torch
import torch.nn as nn
import torch.optim as optim

# ---------------------------------------------------------
# 1. Load your trained model
# ---------------------------------------------------------
model.load_state_dict(torch.load("unlearned_resnet18_cifar100_0forgetclass_model.pth"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ---------------------------------------------------------
# 2. Freeze backbone
# ---------------------------------------------------------
for param in model.parameters():
    param.requires_grad = False

# ---------------------------------------------------------
# 3. Remove original classifier and expose embeddings
# ---------------------------------------------------------
embedding_dim = model.fc.in_features

# Replace classifier with identity
model.fc = nn.Identity()

model.eval()

# ---------------------------------------------------------
# 4. Define a NEW linear classifier
# ---------------------------------------------------------
linear_classifier = nn.Linear(embedding_dim, num_classes).to(device)

# Only train classifier params
optimizer = optim.Adam(linear_classifier.parameters(), lr=1e-3)

criterion = nn.CrossEntropyLoss()

# ---------------------------------------------------------
# 5. Training loop
# ---------------------------------------------------------
epochs = 10

for epoch in range(epochs):

    linear_classifier.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # -------------------------------------------------
        # Extract frozen embeddings
        # -------------------------------------------------
        with torch.no_grad():
            embeddings = model(images)

        # -------------------------------------------------
        # Train linear layer
        # -------------------------------------------------
        logits = linear_classifier(embeddings)

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {running_loss:.4f} "
        f"Acc: {acc:.2f}%"
    )

Epoch [1/10] Loss: 271.3677 Acc: 87.00%


Epoch [2/10] Loss: 66.1561 Acc: 96.35%


Epoch [3/10] Loss: 47.2485 Acc: 97.06%


Epoch [4/10] Loss: 38.2605 Acc: 97.52%


Epoch [5/10] Loss: 33.4096 Acc: 97.69%


Epoch [6/10] Loss: 29.5439 Acc: 97.84%


Epoch [7/10] Loss: 27.0967 Acc: 98.07%


Epoch [8/10] Loss: 24.4646 Acc: 98.20%


Epoch [9/10] Loss: 23.4631 Acc: 98.20%


Epoch [10/10] Loss: 22.6648 Acc: 98.24%


In [18]:
@torch.no_grad()
def evaluate_accuracy(
    backbone: nn.Module,
    classifier: nn.Module,
    dataloader,
    device: torch.device,
) -> float:
    """
    Evaluate accuracy of frozen backbone + linear classifier.

    Parameters
    ----------
    backbone : nn.Module
        Feature extractor model (fc should usually be Identity()).

    classifier : nn.Module
        Linear classifier trained on embeddings.

    dataloader : DataLoader
        Evaluation dataloader.

    device : torch.device
        CUDA or CPU device.

    Returns
    -------
    float
        Accuracy in percentage.
    """

    backbone.eval()
    classifier.eval()

    correct = 0
    total = 0

    for images, labels in dataloader:

        images = images.to(device)
        labels = labels.to(device)

        # Extract embeddings
        embeddings = backbone(images)

        # Class predictions
        logits = classifier(embeddings)

        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    accuracy = 100.0 * correct / total

    return accuracy

In [19]:
print("-------------------After Probing-------------")
print("Forget Train Accuracy::",evaluate_accuracy(model,linear_classifier,train_forget_loader,device))
print("Retain Train Accuracy::",evaluate_accuracy(model,linear_classifier,train_retain_loader,device))
print("Forget Test Accuracy::",evaluate_accuracy(model,linear_classifier,test_forget_loader,device))
print("Retain Test Accuracy::",evaluate_accuracy(model,linear_classifier,test_retain_loader,device))

-------------------After Probing-------------


Forget Train Accuracy:: 97.6


Retain Train Accuracy:: 98.39393939393939


Forget Test Accuracy:: 57.0


Retain Test Accuracy:: 60.57575757575758
